# pipe_nuevo — 03 Escalado: aplica el metodo elegido sobre el cache de 02_FE

Lee `features_sin_escalar_*.parquet` (armado por `02_FE.ipynb`, que ya corrio
la agregacion, shares, lags, vecinos, etc. -- lo pesado) y agrega el escalado
por fila. Al no repetir el trabajo pesado, correr los 4 metodos para comparar
(`mediana`, `media`, `zscore`, `rolling_mean`) tarda segundos en vez de volver
a correr todo el FE cuatro veces.

El metodo elegido se graba en el nombre del parquet de salida y en el
`_features.json` -- ese archivo es el que lee `03_Optuna.ipynb`.


In [ ]:
import json, time
from pathlib import Path

import polars as pl


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    import os
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
RUTA_FE = BUCKET / "datasets_fe"

print(f"BUCKET  : {BUCKET}")
print(f"datasets: {RUTA_FE}")


In [ ]:
PARAM = {
    # ══ Deben coincidir con 02_FE (para leer el cache correcto) ═════════
    'granularidad': 'pc',
    'solo_productos_target': True,
    'horizonte': 2,
    'max_lags': 12,
    'n_vecinos': 3,

    # ══ Escalado ══════════════════════════════════════════════════════
    # 'mediana'      -> B0=0, B1=mediana(ventana)   norm = valor / B1   (robusto a outliers)
    # 'media'        -> B0=0, B1=media(ventana)     norm = valor / B1
    # 'zscore'       -> B0=media, B1=desvio         norm = (valor-B0) / B1
    # 'rolling_mean' -> B0=0, B1=media movil corta  norm = valor / B1   (reacciona
    #                   rapido a series con tendencia; ventana = 'ventana_escalado',
    #                   tiene que existir como columna tn_ma{ventana} en el cache)
    'metodo_escalado': 'mediana',
    'ventana_escalado': 3,

    'semilla': 102191,
}

G = PARAM['granularidad']
H = PARAM['horizonte']
L = PARAM['max_lags']
ES_PC = G == 'pc'

_grp = 'grpClienteProducto' if ES_PC else 'grpProducto'
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE_SIN_ESCALAR = (f"features_sin_escalar_{_grp}{_tgt}_{L}lags_share_{H}h"
                      f"_vec{PARAM['n_vecinos']}.parquet")

_esc = PARAM['metodo_escalado']
NOMBRE = (f"preprocesado_{_grp}{_tgt}_{L}lags_share_{H}h"
          f"_esc{_esc}_vec{PARAM['n_vecinos']}_pipeNuevo.parquet")

print(f"leo cache : {NOMBRE_SIN_ESCALAR}")
print(f"metodo    : {_esc}   (ventana {PARAM['ventana_escalado']} si rolling_mean)")
print(f"salida    : {NOMBRE}")


In [ ]:
t0 = time.time()

df = pl.read_parquet(RUTA_FE / NOMBRE_SIN_ESCALAR)

print(f"cache: {df.height:,} filas x {df.width} columnas")
print(f"[{time.time()-t0:.0f}s]")


### Escalado configurable

Se calcula fila a fila, SOLO con la ventana `tn0, tn_lag1..tn_lag{max_lags}`
(nunca con la clase -> evita leakage), y se guarda `B0`/`B1` por fila para
poder desescalar despues (`desescalar()` es el inverso exacto).


In [ ]:
def escalar_panel(df_in, metodo="media", ventana_rolling=3, max_lags=None):
    """
    Agrega B0, B1, tn0_norm, tn_lag*_norm y clase_tn_norm (si existe) a df_in.

    metodo:
      "mediana"      -> B0=0, B1=mediana(ventana).            norm = valor / B1
      "media"        -> B0=0, B1=media(ventana).               norm = valor / B1
      "zscore"       -> B0=media(ventana), B1=desvio(ventana). norm = (valor-B0) / B1
      "rolling_mean" -> B0=0, B1=media movil de 'ventana_rolling' meses (tn_ma{v}).
                        Reacciona mas rapido que 'media' a series con tendencia.

    La ventana de ajuste (de la que salen B0/B1) es SIEMPRE tn0, tn_lag1..tn_lagN;
    la clase (t+horizonte) nunca participa del ajuste, solo se normaliza con los
    B0/B1 ya calculados -> no hay leakage.
    """
    lag_cols = sorted(
        [c for c in df_in.columns if c == "tn0" or
         (c.startswith("tn_lag") and c[6:].isdigit())],
        key=lambda c: 0 if c == "tn0" else int(c[6:]),
    )
    if max_lags is not None:
        lag_cols = [c for c in lag_cols if c == "tn0" or int(c[6:]) <= max_lags]

    if metodo == "mediana":
        B0 = pl.lit(0.0)
        B1 = pl.concat_list(lag_cols).list.median()
    elif metodo == "media":
        B0 = pl.lit(0.0)
        B1 = pl.mean_horizontal(lag_cols)
    elif metodo == "zscore":
        B0 = pl.mean_horizontal(lag_cols)
        B1 = pl.concat_list(lag_cols).list.std()
    elif metodo == "rolling_mean":
        ma_col = f"tn_ma{ventana_rolling}"
        if ma_col not in df_in.columns:
            raise ValueError(
                f"'{ma_col}' no existe: agregala a PARAM['ventanas_ma'] de 02_FE, o "
                f"pasa 'ventana_rolling' con un valor que ya este calculado."
            )
        B0 = pl.lit(0.0)
        B1 = pl.col(ma_col)
    else:
        raise ValueError(f"metodo no soportado: {metodo}")

    out = df_in.with_columns(B0.alias("B0"), B1.alias("B1"))
    b1_safe = (pl.when((pl.col("B1") == 0) | pl.col("B1").is_null())
                 .then(1.0).otherwise(pl.col("B1")))

    exprs = [((pl.col(c) - pl.col("B0")) / b1_safe).alias(f"{c}_norm") for c in lag_cols]
    if "clase_tn" in out.columns:
        exprs.append(((pl.col("clase_tn") - pl.col("B0")) / b1_safe).alias("clase_tn_norm"))
    return out.with_columns(exprs)


def desescalar(df_in, col_norm, alias="pred_tn"):
    """Inverso exacto de escalar_panel: recupera la escala original (toneladas)."""
    b1_safe = (pl.when((pl.col("B1") == 0) | pl.col("B1").is_null())
                 .then(1.0).otherwise(pl.col("B1")))
    return df_in.with_columns((pl.col(col_norm) * b1_safe + pl.col("B0")).alias(alias))


In [ ]:
t0 = time.time()

df = escalar_panel(df, metodo=PARAM['metodo_escalado'],
                    ventana_rolling=PARAM['ventana_escalado'], max_lags=L)

# chequeo round-trip: desescalar tiene que devolver exactamente clase_tn
_chk = desescalar(df.filter(pl.col("clase_tn").is_not_null()), "clase_tn_norm", alias="clase_tn_rec")
_err = (_chk["clase_tn"] - _chk["clase_tn_rec"]).abs().max()
print(f"escalado '{PARAM['metodo_escalado']}' aplicado: tn0_norm, tn_lag*_norm, clase_tn_norm agregados")
print(f"chequeo round-trip (desescalar(escalar(clase_tn)) == clase_tn): error maximo {_err:.2e}")
assert _err < 1e-6, "el desescalado no reproduce clase_tn: revisar B0/B1"
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
t0 = time.time()

PROHIBIDAS = {"clase_tn", "clase_tn_norm", "periodo_objetivo"}
FEATURES = [c for c in df.columns if c not in PROHIBIDAS]

path_out = RUTA_FE / NOMBRE
df.write_parquet(path_out)

print(f"Guardado: {path_out}")
print(f"  {df.height:,} filas x {df.width} columnas")
print(f"  tamanio en disco: {path_out.stat().st_size / 1e6:.0f} MB")

familias = {
    "shares": [c for c in FEATURES if c.startswith("sh_") and "_d" not in c and "_ma" not in c and "_lag" not in c],
    "lags de share": [c for c in FEATURES if c.startswith("sh_") and "_lag" in c],
    "ma de share": [c for c in FEATURES if c.startswith("sh_") and "_ma" in c],
    "deltas de share": [c for c in FEATURES if c.startswith("sh_") and ("_d1" in c or "_d3" in c or "_dma" in c)],
    "lags de tn": [c for c in FEATURES if c.startswith("tn_lag") and "_norm" not in c],
    "lags de tn normalizados": [c for c in FEATURES if c.startswith("tn_lag") and c.endswith("_norm")],
    "ma de tn/qty": [c for c in FEATURES if "_ma" in c and not c.startswith("sh_")],
    "indices": [c for c in FEATURES if c.startswith("idx_")],
    "escalado": [c for c in FEATURES if c in ("B0", "B1", "tn0_norm")],
    "edad": [c for c in FEATURES if c in ("edad_cliente_producto", "edad_producto")],
    "recencia": [c for c in FEATURES if c == "meses_sin_compra"],
    "peso acumulado": [c for c in FEATURES if c.startswith("peso_")],
    "vecinos": [c for c in FEATURES if c in ("tn_sustitutos_prom", "tn_complementarios_prom")],
    "totales/contexto": [c for c in FEATURES if c.startswith(("tn_prod", "tn_cli", "tn_cat", "tn_mercado", "n_"))],
    "calendario/ciclo": [c for c in FEATURES if c in ("mes_del_anio", "vendio", "frac_meses_con_venta_6",
                                                      "meses_con_venta_3", "tn_pico_hasta_aca")],
}
print("\nFamilias de features:")
_vistas = set()
for k, v in familias.items():
    if v:
        print(f"  {k:24s} {len(v):3d}")
        _vistas |= set(v)
_resto = [c for c in FEATURES if c not in _vistas]
if _resto:
    print(f"  {'otras':24s} {len(_resto):3d}   {_resto[:8]}")

with open(RUTA_FE / NOMBRE.replace(".parquet", "_features.json"), "w", encoding="utf-8") as f:
    json.dump({"nombre": NOMBRE, "param": PARAM, "n_features": len(FEATURES),
               "features": FEATURES,
               "familias": {k: v for k, v in familias.items() if v}},
              f, indent=2, ensure_ascii=False, default=str)

print(f"\n[{time.time()-t0:.0f}s]")
